# EP05. GAIA · SWE-bench를 내 프로젝트에

## 학습 목표

1. **GAIA 벤치마크**의 구조(Level 1~3)와 설계 원칙을 이해한다
2. GAIA 방법론으로 **커스텀 데이터셋**을 직접 제작한다
3. **Langfuse Dataset**으로 벤치마크 데이터를 버전 관리한다
4. **LLM-as-Judge**로 자동 채점 파이프라인을 구현한다
5. **Langfuse score API**로 결과를 기록하고 리더보드를 생성한다

---

> **AI 가이드**: 이 노트북은 순서대로 실행하도록 설계되었습니다.  
> 섹션 1에서 패키지를 설치하고, 섹션 2에서 API 키를 설정한 후 진행하세요.  
> Langfuse 계정이 없다면 [cloud.langfuse.com](https://cloud.langfuse.com)에서 무료로 생성하세요.  
> GAIA 스타일 커스텀 데이터셋은 섹션 3에서 한국어로 직접 정의합니다.  
> 각 코드 셀은 독립적으로 실행 가능하지만, 위에서 아래 순서로 실행하세요.


## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langchain langchain-anthropic langgraph langfuse datasets python-dotenv

In [ ]:
# 설치 확인
import langchain, langgraph, langfuse
print(f'langchain: {langchain.__version__}')
print(f'langgraph: {langgraph.__version__}')
print(f'langfuse:  {langfuse.__version__}')

## 섹션 2. 라이브러리 로드 + Langfuse 초기화

In [ ]:
import os
import time
import json
from dotenv import load_dotenv

load_dotenv()

# API 키 설정 (환경 변수 또는 직접 입력)
# os.environ['ANTHROPIC_API_KEY']   = 'sk-ant-...'
# os.environ['LANGFUSE_PUBLIC_KEY'] = 'pk-lf-...'
# os.environ['LANGFUSE_SECRET_KEY'] = 'sk-lf-...'
# os.environ['LANGFUSE_HOST']       = 'https://cloud.langfuse.com'

from langfuse import Langfuse

langfuse = Langfuse()
langfuse.auth_check()
print('Langfuse 연결 성공!')

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langfuse.langchain import CallbackHandler

print('라이브러리 로드 완료')

## 섹션 3. GAIA 스타일 커스텀 데이터셋 10개 정의

**[커스텀 벤치마크 평가 파이프라인]**
```mermaid
flowchart LR
    A(1️⃣ 요구사항 분석<br/>어떤 능력을 측정?):::step1 --> B(2️⃣ 태스크 설계<br/>문제 유형·레벨 정의):::step2
    B --> C(3️⃣ 평가 기준 확정<br/>채점 방법 자통화):::step3
    C --> D(4️⃣ 자동화 구현<br/>Langfuse Dataset+Experiment):::step4
    classDef step1 fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef step2 fill:#fff3e0,stroke:#fb8c00,stroke-width:2px,color:#000
    classDef step3 fill:#fce4ec,stroke:#d81b60,stroke-width:2px,color:#000
    classDef step4 fill:#e8f5e9,stroke:#43a047,stroke-width:2px,color:#000
```

GAIA 벤치마크의 Level 1~3 구조를 따르는 한국어 커스텀 문제입니다.  
각 문제는 `question`, `answer`, `level`, `category`, `required_tools` 필드를 포함합니다.

| 레벨 | 설명 | 필요 능력 |
|------|------|----------|
| Level 1 | 단일 도구, 직접 답변 | 계산기 또는 상식 |
| Level 2 | 다단계, 복합 도구 | 검색 + 계산 조합 |
| Level 3 | 복합 추론, 다중 소스 | 여러 도구 + 논리 |


In [ ]:
gaia_dataset = [
    # ====== Level 1: 단일 도구로 해결 ======
    {
        'id': 'GAIA-001',
        'question': '지구에서 달까지의 평균 거리는 약 몇 킬로미터인가요?',
        'answer': '384400',
        'level': 1,
        'category': 'science',
        'required_tools': ['search'],
    },
    {
        'id': 'GAIA-002',
        'question': '대한민국 헌법 제1조 1항의 내용은 무엇인가요?',
        'answer': '대한민국은 민주공화국이다',
        'level': 1,
        'category': 'korean_law',
        'required_tools': ['search'],
    },
    {
        'id': 'GAIA-003',
        'question': '2의 10제곱은 얼마인가요?',
        'answer': '1024',
        'level': 1,
        'category': 'math',
        'required_tools': ['calculator'],
    },
    {
        'id': 'GAIA-004',
        'question': '현재 FIFA 월드컵 최다 우승국은 어느 나라인가요?',
        'answer': '브라질',
        'level': 1,
        'category': 'sports',
        'required_tools': ['search'],
    },
    # ====== Level 2: 다단계 처리 필요 ======
    {
        'id': 'GAIA-005',
        'question': '서울에서 부산까지의 직선 거리(약 325km)를 시속 300km 고속철도로 이동하면 몇 분이 걸리나요?',
        'answer': '65',
        'level': 2,
        'category': 'math',
        'required_tools': ['calculator'],
    },
    {
        'id': 'GAIA-006',
        'question': '조선왕조의 존속 기간(년)을 계산하세요. (1392년 건국, 1897년 대한제국 선포)',
        'answer': '505',
        'level': 2,
        'category': 'korean_history',
        'required_tools': ['calculator'],
    },
    {
        'id': 'GAIA-007',
        'question': '한국의 주요 3대 산맥(태백산맥, 소백산맥, 낭림산맥) 중 가장 긴 것은 무엇이며, 그 길이는?',
        'answer': '태백산맥',
        'level': 2,
        'category': 'korean_geography',
        'required_tools': ['search'],
    },
    {
        'id': 'GAIA-008',
        'question': '파이썬에서 리스트 [3, 1, 4, 1, 5, 9, 2, 6]의 중앙값(median)을 구하세요.',
        'answer': '3.5',
        'level': 2,
        'category': 'programming',
        'required_tools': ['calculator'],
    },
    # ====== Level 3: 복합 추론 필요 ======
    {
        'id': 'GAIA-009',
        'question': '어떤 회사의 2022년 매출이 100억이고 연평균 성장률이 15%라면, 2025년 예상 매출은 얼마인가요? (억 원, 소수점 두 자리)',
        'answer': '152.09',
        'level': 3,
        'category': 'finance',
        'required_tools': ['calculator'],
    },
    {
        'id': 'GAIA-010',
        'question': '세종대왕이 훈민정음을 창제한 해(1443년)부터 현재(2025년)까지 몇 년이 지났으며, '
                    '이는 1000년의 몇 퍼센트인가요? (소수점 한 자리)',
        'answer': '58.2',
        'level': 3,
        'category': 'korean_history',
        'required_tools': ['calculator'],
    },
]

print(f'데이터셋 총 {len(gaia_dataset)}개 문제 정의 완료')
print()
for item in gaia_dataset:
    print(f"  {item['id']} | Level {item['level']} | {item['category']} | {item['question'][:50]}")

## 섹션 4. 데이터셋 구조 확인 및 분포 분석

데이터셋의 레벨별, 카테고리별 분포를 확인합니다.


In [ ]:
import pandas as pd

df_dataset = pd.DataFrame(gaia_dataset)

print('=== 데이터셋 구조 ===\n')
print(f'필드: {list(df_dataset.columns)}')
print()
print('레벨별 분포:')
print(df_dataset.groupby('level').size().to_string())
print()
print('카테고리별 분포:')
print(df_dataset.groupby('category').size().to_string())
print()
print('필요 도구별 분포:')
tools_flat = [t for tools in df_dataset['required_tools'] for t in tools]
from collections import Counter
print(dict(Counter(tools_flat)))
print()
display(df_dataset[['id', 'level', 'category', 'question', 'answer', 'required_tools']])

## 섹션 5. Langfuse Dataset 생성 및 업로드

커스텀 GAIA 데이터셋을 Langfuse에 업로드합니다.  
이후 모든 실험은 이 데이터셋을 기준으로 실행됩니다.


In [ ]:
DATASET_NAME = 'ep05-custom-gaia-v1'

try:
    dataset = langfuse.get_dataset(DATASET_NAME)
    print(f'기존 데이터셋 로드: {DATASET_NAME}')
except Exception:
    dataset = langfuse.create_dataset(
        name=DATASET_NAME,
        description='EP05 GAIA 스타일 한국어 커스텀 벤치마크 v1.0'
    )
    print(f'새 데이터셋 생성: {DATASET_NAME}')

for item in gaia_dataset:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={
            'question': item['question'],
            'level': item['level'],
        },
        expected_output={'answer': item['answer']},
        metadata={
            'id': item['id'],
            'level': item['level'],
            'category': item['category'],
            'required_tools': item['required_tools'],
        }
    )

langfuse.flush()
print(f'\n{len(gaia_dataset)}개 문제 업로드 완료')
print(f'Langfuse → Datasets → {DATASET_NAME}')

## 섹션 6. 평가용 에이전트 구축

계산기 도구와 검색 목(Mock) 도구를 포함한 LangGraph 에이전트를 만듭니다.  
실제 환경에서는 `search_mock`을 실제 검색 도구(Tavily 등)로 교체하세요.


In [ ]:
import math
from statistics import median

@tool
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. Python 수식으로 입력하세요.
    예: '2**10', '325/300*60', '100 * 1.15**3'
    """
    try:
        allowed = {
            'math': math, 'sum': sum, 'range': range,
            'abs': abs, 'round': round, 'median': median,
            'sorted': sorted, 'len': len,
        }
        result = eval(expression, {'__builtins__': {}}, allowed)
        return str(result)
    except Exception as e:
        return f'계산 오류: {e}'


@tool
def search_mock(query: str) -> str:
    """검색 도구 (Mock). 실제 환경에서는 Tavily/SerpAPI로 교체하세요."""
    knowledge_base = {
        '달까지 거리': '지구에서 달까지의 평균 거리는 약 384,400km입니다.',
        '헌법 1조': '대한민국 헌법 제1조 1항: 대한민국은 민주공화국이다.',
        'FIFA 월드컵 최다': 'FIFA 월드컵 최다 우승국은 브라질로 5회 우승했습니다.',
        '태백산맥': '태백산맥은 한국에서 가장 긴 산맥으로 약 500km 이상 뻗어 있습니다.',
    }
    for key, value in knowledge_base.items():
        if key in query:
            return value
    return f'검색 결과: [{query}]에 대한 정보를 찾았습니다. (Mock 응답)'


def build_agent(model_name: str):
    """지정된 모델로 ReAct 에이전트를 생성합니다."""
    model = ChatAnthropic(model=model_name, temperature=0)
    return create_react_agent(model, tools=[calculator, search_mock])


agent_haiku = build_agent('claude-haiku-4-5')
print('에이전트 구축 완료')

# 빠른 동작 테스트
test = agent_haiku.invoke({'messages': [('user', '2의 10제곱은?')]})
print('테스트 응답:', test['messages'][-1].content[:60])

## 섹션 7. 자동 평가 실행 함수 (run_benchmark)

Langfuse Dataset을 기반으로 벤치마크를 실행합니다.  
각 실행 결과는 Langfuse 트레이스에 자동으로 기록됩니다.


In [ ]:
def exact_match_score(answer: str, expected: str) -> float:
    """정답 포함 여부로 빠르게 채점 (LLM-as-Judge 호출 전 1차 필터)."""
    a = answer.strip().lower().replace(',', '')
    e = expected.strip().lower().replace(',', '')
    if e in a:
        return 1.0
    try:
        a_num = float(a.split()[0])
        e_num = float(e.split()[0])
        if abs(a_num - e_num) / (abs(e_num) + 1e-9) < 0.02:
            return 1.0
    except Exception:
        pass
    return 0.0


def run_benchmark(
    agent,
    dataset_name: str,
    run_name: str,
    model_name: str = 'unknown',
    use_llm_judge: bool = True,
) -> pd.DataFrame:
    """벤치마크를 실행하고 결과 DataFrame을 반환합니다."""
    dataset = langfuse.get_dataset(dataset_name)
    results = []

    for item in dataset.items:
        question = item.input['question']
        expected = item.expected_output['answer']
        tc_id    = item.metadata.get('id', '?')
        level    = item.metadata.get('level', 0)
        category = item.metadata.get('category', '?')

        print(f'  [{tc_id}] L{level} 실행 중...', end=' ')
        start = time.time()

        with item.observe(run_name=run_name) as trace:
            try:
                langfuse_cb = CallbackHandler()
                response = agent.invoke(
                    {'messages': [('user', question)]},
                    config={'recursion_limit': 20, 'callbacks': [langfuse_cb]}
                )
                answer  = response['messages'][-1].content
                latency = time.time() - start

                # 1차: Exact Match
                em_score = exact_match_score(answer, expected)
                # 2차: LLM-as-Judge (옵션)
                judge_score = em_score  # LLM Judge 없이는 EM 사용
                status = 'PASS' if em_score >= 0.5 else 'FAIL'

                trace.score(name='exact_match',      value=em_score)
                trace.score(name='judge_score',       value=judge_score)
                trace.score(name='latency_seconds',   value=round(latency, 2))

            except Exception as exc:
                answer  = f'ERROR: {exc}'
                latency = time.time() - start
                em_score = judge_score = 0.0
                status = 'ERROR'

        results.append({
            'id': tc_id, 'level': level, 'category': category,
            'question': question[:60], 'expected': expected,
            'answer': answer[:80], 'exact_match': em_score,
            'judge_score': judge_score, 'latency': round(latency, 2),
            'status': status, 'model': model_name, 'run': run_name,
        })
        print(f'{status} (em={em_score:.1f}, {latency:.1f}s)')

    langfuse.flush()
    return pd.DataFrame(results)

print('run_benchmark 함수 정의 완료')

## 섹션 8. LLM-as-Judge 평가 구현

Exact Match로 채점하기 어려운 주관식 응답을 LLM이 자동으로 채점합니다.  
채점 프롬프트를 잘 설계하는 것이 핵심입니다.


In [ ]:
judge_model = ChatAnthropic(model='claude-haiku-4-5', temperature=0)

JUDGE_SYSTEM_PROMPT = """당신은 AI 에이전트 응답을 평가하는 공정한 채점자입니다.
주어진 기준 정답과 에이전트 응답을 비교하여 0.0~1.0 사이의 점수를 부여합니다.
- 1.0: 정확히 동일하거나 의미적으로 완전히 일치
- 0.5: 부분적으로 정확 (핵심 수치/사실은 맞지만 형식 다름)
- 0.0: 오답 또는 관련 없는 응답
점수(숫자 하나)만 출력하세요."""


def llm_judge(question: str, expected: str, answer: str) -> float:
    """LLM-as-Judge: 응답 품질을 0.0~1.0으로 채점합니다."""
    user_prompt = (
        f'질문: {question}\n'
        f'기준 정답: {expected}\n'
        f'에이전트 응답: {answer}\n\n'
        f'점수 (0.0~1.0):'
    )
    try:
        response = judge_model.invoke([
            SystemMessage(content=JUDGE_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ])
        score_text = response.content.strip().split()[0]
        score = float(score_text)
        return max(0.0, min(1.0, score))
    except Exception:
        return 0.0


# LLM-as-Judge 테스트
test_q  = '서울에서 부산까지 고속철도로 몇 분 걸리나요?'
test_e  = '65'
test_a1 = '약 65분 정도 걸립니다.'
test_a2 = '서울에서 부산까지는 버스로 5시간 걸립니다.'

score1 = llm_judge(test_q, test_e, test_a1)
score2 = llm_judge(test_q, test_e, test_a2)

print(f'정답에 가까운 응답 점수: {score1}')
print(f'오답 응답 점수:          {score2}')

## 섹션 9. Langfuse score API로 결과 기록

LLM-as-Judge 점수를 포함한 완전한 벤치마크를 실행하고  
모든 결과를 Langfuse score API로 기록합니다.


In [ ]:
def run_full_benchmark(
    agent,
    dataset_name: str,
    run_name: str,
    model_name: str = 'unknown',
) -> pd.DataFrame:
    """Exact Match + LLM-as-Judge 점수를 모두 Langfuse에 기록하는 벤치마크."""
    dataset = langfuse.get_dataset(dataset_name)
    results = []

    for item in dataset.items:
        question = item.input['question']
        expected = item.expected_output['answer']
        tc_id    = item.metadata.get('id', '?')
        level    = item.metadata.get('level', 0)
        category = item.metadata.get('category', '?')

        print(f'  [{tc_id}] L{level} 실행 중...', end=' ')
        start = time.time()

        with item.observe(run_name=run_name) as trace:
            try:
                langfuse_cb = CallbackHandler()
                response = agent.invoke(
                    {'messages': [('user', question)]},
                    config={'recursion_limit': 20, 'callbacks': [langfuse_cb]}
                )
                answer  = response['messages'][-1].content
                latency = time.time() - start

                em_score    = exact_match_score(answer, expected)
                judge_score = llm_judge(question, expected, answer)
                final_score = max(em_score, judge_score)
                status = 'PASS' if final_score >= 0.5 else 'FAIL'

                # Langfuse score API로 상세 기록
                trace.score(name='exact_match',    value=em_score,
                             comment=f'expected={expected}')
                trace.score(name='llm_judge',       value=judge_score,
                             comment=f'judge evaluated: {answer[:30]}')
                trace.score(name='final_score',     value=final_score)
                trace.score(name='latency_seconds', value=round(latency, 2))
                trace.score(name='level',           value=float(level))

            except Exception as exc:
                answer  = f'ERROR: {exc}'
                latency = time.time() - start
                em_score = judge_score = final_score = 0.0
                status = 'ERROR'

        results.append({
            'id': tc_id, 'level': level, 'category': category,
            'question': question[:60], 'expected': expected,
            'answer': answer[:80], 'exact_match': em_score,
            'llm_judge': judge_score, 'final_score': final_score,
            'latency': round(latency, 2), 'status': status,
            'model': model_name, 'run': run_name,
        })
        print(f'{status} (em={em_score:.1f}, judge={judge_score:.1f}, {latency:.1f}s)')

    langfuse.flush()
    return pd.DataFrame(results)


print('run_full_benchmark 함수 정의 완료')

In [ ]:
# 실험 A: claude-haiku-4-5
print('=== 실험 A: claude-haiku-4-5 ===\n')
df_haiku = run_full_benchmark(
    agent=build_agent('claude-haiku-4-5'),
    dataset_name=DATASET_NAME,
    run_name='benchmark-haiku',
    model_name='claude-haiku-4-5'
)
print()

# 실험 B: claude-3-5-sonnet-20241022
print('=== 실험 B: claude-3-5-sonnet-20241022 ===\n')
df_sonnet = run_full_benchmark(
    agent=build_agent('claude-3-5-sonnet-20241022'),
    dataset_name=DATASET_NAME,
    run_name='benchmark-sonnet',
    model_name='claude-3-5-sonnet-20241022'
)
print('\n두 실험 완료!')

## 섹션 10. 리더보드 DataFrame 생성 및 시각화

In [ ]:
def build_leaderboard(dfs: list[pd.DataFrame]) -> pd.DataFrame:
    """여러 실험 결과로 리더보드 DataFrame을 생성합니다."""
    rows = []
    for df in dfs:
        rows.append({
            'model':            df['model'].iloc[0],
            'overall_score':    round(df['final_score'].mean(), 4),
            'exact_match':      round(df['exact_match'].mean(), 4),
            'llm_judge':        round(df['llm_judge'].mean(), 4),
            'pass_rate':        f"{df['status'].eq('PASS').mean():.1%}",
            'avg_latency':      round(df['latency'].mean(), 2),
            'level1_score':     round(df[df['level']==1]['final_score'].mean(), 3),
            'level2_score':     round(df[df['level']==2]['final_score'].mean(), 3),
            'level3_score':     round(df[df['level']==3]['final_score'].mean(), 3),
            'total_cases':      len(df),
        })
    lb = pd.DataFrame(rows).sort_values('overall_score', ascending=False).reset_index(drop=True)
    lb.index = [f'{i+1}위' for i in range(len(lb))]
    return lb


leaderboard = build_leaderboard([df_sonnet, df_haiku])

print('=' * 65)
print('                   GAIA 커스텀 벤치마크 리더보드')
print('=' * 65)
display(leaderboard)

print('\n--- 레벨별 상세 비교 ---')
combined = pd.concat([df_haiku, df_sonnet])
display(combined.groupby(['model', 'level'])['final_score'].mean().unstack().round(3))

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['font.family'] = 'AppleGothic'

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = ['#5b9bd5', '#ed7d31']
    models = ['haiku-4-5', 'sonnet-20241022']

    # 1) 전체 점수 비교
    overall = [df_haiku['final_score'].mean(), df_sonnet['final_score'].mean()]
    axes[0, 0].bar(models, overall, color=colors)
    axes[0, 0].set_title('전체 최종 점수'); axes[0, 0].set_ylim(0, 1)

    # 2) EM vs Judge 비교
    x, w = range(2), 0.35
    em_scores = [df_haiku['exact_match'].mean(), df_sonnet['exact_match'].mean()]
    jg_scores = [df_haiku['llm_judge'].mean(),   df_sonnet['llm_judge'].mean()]
    axes[0, 1].bar([i - w/2 for i in x], em_scores, w, label='Exact Match', color='#5b9bd5')
    axes[0, 1].bar([i + w/2 for i in x], jg_scores, w, label='LLM Judge',   color='#ed7d31')
    axes[0, 1].set_title('Exact Match vs LLM Judge')
    axes[0, 1].set_xticks(x); axes[0, 1].set_xticklabels(models)
    axes[0, 1].legend(); axes[0, 1].set_ylim(0, 1)

    # 3) 레벨별 성공률
    lvl_data = combined.groupby(['model', 'level'])['final_score'].mean().unstack().T
    xi = range(len(lvl_data))
    axes[1, 0].bar([i - w/2 for i in xi], lvl_data.iloc[:, 0], w, label=models[0], color=colors[0])
    axes[1, 0].bar([i + w/2 for i in xi], lvl_data.iloc[:, 1], w, label=models[1], color=colors[1])
    axes[1, 0].set_title('레벨별 점수'); axes[1, 0].set_ylim(0, 1)
    axes[1, 0].set_xticks(xi); axes[1, 0].set_xticklabels([f'Level {i+1}' for i in xi])
    axes[1, 0].legend()

    # 4) 응답 시간
    latencies = [df_haiku['latency'].mean(), df_sonnet['latency'].mean()]
    axes[1, 1].bar(models, latencies, color=colors)
    axes[1, 1].set_title('평균 응답 시간 (초)')

    plt.tight_layout()
    plt.savefig('ep05_benchmark_results.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('그래프 저장: ep05_benchmark_results.png')
except ImportError:
    print('matplotlib 미설치 — 텍스트 결과만 표시')

## 섹션 11. 레벨별 · 카테고리별 성능 분석

어느 레벨, 어느 카테고리에서 에이전트가 약한지 분석합니다.  
이 분석을 바탕으로 프롬프트나 도구를 개선할 수 있습니다.


In [ ]:
print('=== 레벨별 성능 분석 ===\n')
level_analysis = combined.groupby(['level', 'model']).agg(
    avg_score=('final_score', 'mean'),
    pass_rate=('status', lambda x: x.eq('PASS').mean()),
    avg_latency=('latency', 'mean'),
    count=('id', 'count')
).round(3)
display(level_analysis)

print()
print('=== 카테고리별 성능 분석 ===\n')
cat_analysis = combined.groupby(['category', 'model']).agg(
    avg_score=('final_score', 'mean'),
    pass_rate=('status', lambda x: x.eq('PASS').mean()),
    count=('id', 'count')
).round(3)
display(cat_analysis)

print()
print('=== 약점 분석 (점수 0.5 미만 케이스) ===\n')
weak_cases = combined[combined['final_score'] < 0.5][
    ['id', 'level', 'category', 'model', 'question', 'expected', 'answer', 'final_score']
].sort_values(['level', 'final_score'])
display(weak_cases)

In [ ]:
# 모델 선택 가이드라인 출력
haiku_overall  = df_haiku['final_score'].mean()
sonnet_overall = df_sonnet['final_score'].mean()
haiku_latency  = df_haiku['latency'].mean()
sonnet_latency = df_sonnet['latency'].mean()

print('=' * 55)
print('모델 선택 가이드라인')
print('=' * 55)
print(f'  Haiku  점수: {haiku_overall:.2%}  | 응답시간: {haiku_latency:.2f}초')
print(f'  Sonnet 점수: {sonnet_overall:.2%}  | 응답시간: {sonnet_latency:.2f}초')
print()

score_diff = sonnet_overall - haiku_overall
latency_diff = sonnet_latency - haiku_latency

if score_diff > 0.1:
    print('결론: 정확도 차이가 크므로 Sonnet 권장 (정확도 우선 서비스)')
elif score_diff > 0.05:
    print('결론: 용도에 따라 선택. 비용 중시→Haiku, 정확도 중시→Sonnet')
else:
    print('결론: 성능 차이가 미미하므로 비용이 낮은 Haiku 권장')

## Exercise

### Exercise 1: GAIA 스타일 5문제 커스텀 데이터셋 작성

아래 형식으로 Level 1~3 문제를 각각 작성하세요 (Level 1: 2개, Level 2: 2개, Level 3: 1개).

```python
{
    'id': 'MY-001',
    'question': '...',
    'answer': '...',
    'level': 1,
    'category': 'math|reasoning|search|history',
    'required_tools': ['calculator']
}
```

**목표**: 각 문제에 "왜 이 레벨인가?" 주석 작성 + Langfuse Dataset 업로드

---

### Exercise 2: Langfuse로 리더보드 구축

Exercise 1의 데이터셋으로 두 실험을 실행하세요.

- 실험 A: `claude-haiku-4-5` + LLM-as-Judge
- 실험 B: `claude-3-5-sonnet-20241022` + LLM-as-Judge

**목표**: 레벨별 점수가 포함된 리더보드 DataFrame 생성 + 최종 모델 선택 근거 작성


In [ ]:
# Exercise 1 — 직접 작성하세요
my_gaia_items = [
    # Level 1 문제 2개
    # {'id': 'MY-001', 'question': '...', 'answer': '...', 'level': 1,
    #  'category': 'math', 'required_tools': ['calculator']},
    # ...
    # Level 2 문제 2개
    # Level 3 문제 1개
]

if my_gaia_items:
    MY_DATASET = 'ep05-exercise-v1'
    try:
        langfuse.get_dataset(MY_DATASET)
    except Exception:
        langfuse.create_dataset(name=MY_DATASET, description='Exercise 1 데이터셋')
    for item in my_gaia_items:
        langfuse.create_dataset_item(
            dataset_name=MY_DATASET,
            input={'question': item['question'], 'level': item['level']},
            expected_output={'answer': item['answer']},
            metadata={'id': item['id'], 'level': item['level'],
                      'category': item['category'], 'required_tools': item['required_tools']}
        )
    langfuse.flush()
    print(f'{len(my_gaia_items)}개 문제 업로드 완료: {MY_DATASET}')
else:
    print('my_gaia_items 리스트를 채워주세요!')